In [5]:
from __future__ import annotations

import json
from pathlib import Path

data_path = Path("/zfsauton/scratch/mineuih/waymax_rs/instructions/training/tfrecord_0.jsonl")
index_path = Path("/zfsauton/scratch/mineuih/waymax_rs/instructions/training/tfrecord_0_index.json")

scenario_index = 15
wanted_timestep = 10

key = f"{scenario_index}:{wanted_timestep}"
index = json.loads(index_path.read_text(encoding="utf-8"))
offsets = index.get(key, [])
print(offsets)

instructions = []
with data_path.open("r", encoding="utf-8") as f:
    for offset in offsets:
        f.seek(offset)
        line = f.readline()
        if not line:
            continue
        obj = json.loads(line)
        print(obj)

# print(instructions)

[62615, 62855, 63098, 63329, 63560]
{'scenario_index': 15, 'timestep': 10, 'summary': 'Ego vehicle 0 follows a straight path forward, as indicated by its blue dashed future trajectory.', 'risks': 'none visible', 'instruction': 'Go slightly to the right while moving forward'}
{'scenario_index': 15, 'timestep': 10, 'summary': 'The ego vehicle 0 follows a straight path forward, but its future trajectory is hidden or ambiguous.', 'risks': 'none visible', 'instruction': 'Go slightly to the right while moving forward'}
{'scenario_index': 15, 'timestep': 10, 'summary': 'The ego vehicle 0 follows a straight path forward, as indicated by the blue dashed line.', 'risks': 'none visible', 'instruction': 'Go slightly to the right while moving forward'}
{'scenario_index': 15, 'timestep': 10, 'summary': 'The ego vehicle 0 follows a straight path forward, as indicated by the blue dashed line.', 'risks': 'none visible', 'instruction': 'Go slightly to the right while moving forward'}
{'scenario_index': 

In [6]:
from __future__ import annotations

import json
from collections import defaultdict
from pathlib import Path

training_dir = Path("/zfsauton/scratch/mineuih/waymax_rs/instructions/training")
jsonl_paths = sorted(training_dir.glob("tfrecord_*.jsonl"))

for data_path in jsonl_paths:
    index: dict[str, list[int]] = defaultdict(list)

    with data_path.open("rb") as f:
        while True:
            offset = f.tell()
            line = f.readline()
            if not line:
                break

            line = line.strip()
            if not line:
                continue

            try:
                obj = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f"Failed to parse JSON line in {data_path} at byte offset {offset}") from exc

            scenario_index = obj.get("scenario_index")
            timestep = obj.get("timestep")
            if scenario_index is None or timestep is None:
                continue

            key = f"{scenario_index}:{timestep}"
            index[key].append(offset)

    index_path = data_path.with_name(f"{data_path.stem}_index.json")
    with index_path.open("w", encoding="utf-8") as f:
        json.dump(index, f, ensure_ascii=False, indent=2, sort_keys=True)

    print(f"wrote {index_path} ({len(index)} keys)")


wrote /zfsauton/scratch/mineuih/waymax_rs/instructions/training/tfrecord_0_index.json (1820 keys)
wrote /zfsauton/scratch/mineuih/waymax_rs/instructions/training/tfrecord_1_index.json (1916 keys)
wrote /zfsauton/scratch/mineuih/waymax_rs/instructions/training/tfrecord_10_index.json (1904 keys)
wrote /zfsauton/scratch/mineuih/waymax_rs/instructions/training/tfrecord_100_index.json (2040 keys)
wrote /zfsauton/scratch/mineuih/waymax_rs/instructions/training/tfrecord_101_index.json (1980 keys)
wrote /zfsauton/scratch/mineuih/waymax_rs/instructions/training/tfrecord_102_index.json (1840 keys)
wrote /zfsauton/scratch/mineuih/waymax_rs/instructions/training/tfrecord_103_index.json (1860 keys)
wrote /zfsauton/scratch/mineuih/waymax_rs/instructions/training/tfrecord_104_index.json (1868 keys)
wrote /zfsauton/scratch/mineuih/waymax_rs/instructions/training/tfrecord_105_index.json (2068 keys)
wrote /zfsauton/scratch/mineuih/waymax_rs/instructions/training/tfrecord_106_index.json (1992 keys)
wrote

In [12]:
import numpy as np
file_path = Path("/zfsauton/scratch/mineuih/waymax_rs/cache/training_tfexample.tfrecord-00999-of-01000_t30.npz")
data = np.load(file_path)
# show the keys in the data
print(data.keys())

print(data['features/inst_features_multi'].shape)

KeysView(NpzFile '/zfsauton/scratch/mineuih/waymax_rs/cache/training_tfexample.tfrecord-00999-of-01000_t30.npz' with keys: features/ego_state, features/ego_trajectory, features/goal_xy, features/remaining_timesteps, features/other_states...)
(442, 5, 768)
